# Hotel Booking Intelligence — Colab launcher

This notebook runs the shared Python package. It currently validates setup and initializes run metadata only; training comes later. Supply your own H2.csv. Choose a CPU runtime initially.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPOSITORY = "https://github.com/OMZaky/hotel-booking-intelligence.git"
REVISION = "main"  # Replace with a commit SHA for a reproducible experiment.
WORKDIR = Path("/content/hotel-booking-intelligence")

if not (3, 12) <= sys.version_info[:2] < (3, 15):
    raise RuntimeError("This package supports Python 3.12–3.14. Check your Colab runtime.")
if not WORKDIR.exists():
    subprocess.run(["git", "clone", REPOSITORY, str(WORKDIR)], check=True)
else:
    remote = subprocess.check_output(
        ["git", "-C", str(WORKDIR), "remote", "get-url", "origin"], text=True
    ).strip()
    if remote != REPOSITORY:
        raise RuntimeError("Existing checkout has another origin; choose a new WORKDIR.")
    dirty = subprocess.check_output(
        ["git", "-C", str(WORKDIR), "status", "--porcelain"], text=True
    ).strip()
    if dirty:
        raise RuntimeError("Checkout has changes. Save/review them before switching revisions.")
subprocess.run(["git", "-C", str(WORKDIR), "fetch", "origin"], check=True)
# The main setup default resolves to the latest fetched main, not a stale local branch.
resolved_revision = "origin/main" if REVISION == "main" else REVISION
subprocess.run(["git", "-C", str(WORKDIR), "checkout", "--detach", resolved_revision], check=True)
os.chdir(WORKDIR)
print("Code SHA:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".[ml]"], check=True)
subprocess.run([sys.executable, "-m", "hotel_ml", "doctor"], check=True)

## Optional: persist runs with Google Drive

Put H2.csv at `MyDrive/hotel-booking-intelligence/data/H2.csv`. This cell copies it into runtime storage for reads and saves experiment outputs directly to Drive. Colab will ask you to authorize Drive access. Run it only if you want that access; alternatively upload H2.csv manually into `data/raw/` and copy outputs to persistent storage yourself.


In [ ]:
import shutil

from google.colab import drive

drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/hotel-booking-intelligence")
SOURCE_DATA = DRIVE_ROOT / "data" / "H2.csv"
LOCAL_DATA = WORKDIR / "data" / "raw" / "H2.csv"
if not SOURCE_DATA.is_file():
    raise FileNotFoundError(f"Provide your H2.csv at {SOURCE_DATA}")
LOCAL_DATA.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(SOURCE_DATA, LOCAL_DATA)
os.environ["HOTEL_ML_DATA_PATH"] = str(LOCAL_DATA)
os.environ["HOTEL_ML_ARTIFACT_ROOT"] = str(DRIVE_ROOT / "artifacts")
print("Dataset copied; artifact output will persist in Drive.")

## Initialize an experiment record

Choose one of the starter recipes. These are setup examples with unresolved feature/split policies. `init-run` records provenance, without training. The dataset hash is added when your H2.csv is present. It does not verify the dataset schema or scientific validity.


In [ ]:
CONFIG = "configs/regression/v1/ridge-baseline.json"
DATA = Path(os.environ.get("HOTEL_ML_DATA_PATH", "data/raw/H2.csv"))
if not DATA.is_file():
    raise FileNotFoundError("Supply H2.csv or run the Drive cell before initializing this run.")
if not os.environ.get("HOTEL_ML_ARTIFACT_ROOT"):
    raise RuntimeError("Set HOTEL_ML_ARTIFACT_ROOT to persistent storage before creating a run.")
subprocess.run(
    [sys.executable, "-m", "hotel_ml", "validate-config", "--config", CONFIG], check=True
)
subprocess.run(
    [sys.executable, "-m", "hotel_ml", "init-run", "--config", CONFIG, "--data", str(DATA)],
    check=True,
)

Training, tuning, evaluation and inference cells will be added only after those package commands exist. Review the roadmap for phases. Keep outputs cleared when committing this notebook.
